# Visual Data Analysis with Python
**Portfolio project | June 2025 – August 2025**

## Objective
Explore a 12-feature dataset using univariate, bivariate and multivariate visual analysis; detect unusual observations; study feature correlations; and reduce dimensionality using PCA.

> **Dataset note:** This project uses a reproducible synthetic dataset. The injected anomaly flag is included only for project validation; do not use it as an input to the unsupervised anomaly detector. The analysis computes its findings from the generated data.

In [ ]:
# Install dependencies if needed (run once in Google Colab)
# !pip install pandas numpy matplotlib seaborn plotly scikit-learn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import classification_report, confusion_matrix

sns.set_theme(style="whitegrid")
df = pd.read_csv("visual_analysis_dataset.csv")
feature_cols = [c for c in df.columns if c.startswith("feature_")]
X = df[feature_cols]

print("Shape:", df.shape)
display(df.head())
display(df[feature_cols].describe().T)

## 1. Data quality checks
Check missing values, duplicate rows and data types before analysis.

In [ ]:
print("Missing values:", int(df.isna().sum().sum()))
print("Duplicate rows:", int(df.duplicated().sum()))
print(df.dtypes)

## 2. Univariate analysis
Use histograms and box plots to understand distributions, spread and potential extreme values.

In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(16, 13))
for col, ax in zip(feature_cols, axes.flat):
    sns.histplot(df[col], kde=True, ax=ax)
    ax.set_title(f"Distribution: {col}")
plt.tight_layout()
plt.show()

plt.figure(figsize=(16, 6))
sns.boxplot(data=X, orient="h")
plt.title("Feature-wise box plots")
plt.tight_layout()
plt.show()

## 3. Bivariate and multivariate analysis
Examine pairwise relationships and the correlation structure across all 12 features.

In [ ]:
corr = X.corr()
plt.figure(figsize=(12, 9))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Feature correlation heatmap")
plt.tight_layout()
plt.show()

# Interactive pairwise view using a manageable sample
sample = df.sample(min(400, len(df)), random_state=42)
fig = px.scatter_matrix(sample, dimensions=feature_cols[:6], color="segment",
                        title="Scatter-matrix: first six features")
fig.update_traces(diagonal_visible=False, showupperhalf=False)
fig.show()

upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
top_pairs = upper.stack().abs().sort_values(ascending=False).head(10)
display(top_pairs.rename("absolute_correlation").to_frame())

## 4. Outlier detection with Isolation Forest
Fit the model only on standardized feature values. The `known_injected_anomaly` column is excluded from training and is used only for an optional validation comparison.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

iso = IsolationForest(contamination=0.12, random_state=42)
df["outlier_prediction"] = (iso.fit_predict(X_scaled) == -1).astype(int)
outlier_rate = df["outlier_prediction"].mean() * 100
print(f"Detected outliers: {df['outlier_prediction'].sum()} / {len(df)} ({outlier_rate:.1f}%)")

# Compare with synthetic injection labels for a sanity check only
print(pd.crosstab(df["known_injected_anomaly"], df["outlier_prediction"],
                  rownames=["Injected anomaly"], colnames=["Detected anomaly"]))

## 5. PCA: reduce 12 features to 2 components
Standardize first because PCA is scale-sensitive. Inspect explained variance before interpreting the two-dimensional projection.

In [ ]:
pca_full = PCA(random_state=42)
pca_full.fit(X_scaled)
cum_var = np.cumsum(pca_full.explained_variance_ratio_)

plt.figure(figsize=(8, 5))
plt.plot(range(1, len(cum_var)+1), cum_var, marker="o")
plt.axhline(0.85, linestyle="--", label="85% variance target")
plt.xticks(range(1, 13))
plt.xlabel("Number of principal components")
plt.ylabel("Cumulative explained variance")
plt.title("PCA explained variance")
plt.legend()
plt.tight_layout()
plt.show()

pca2 = PCA(n_components=2, random_state=42)
components = pca2.fit_transform(X_scaled)
pca_df = pd.DataFrame(components, columns=["PC1", "PC2"])
pca_df["segment"] = df["segment"].astype(str)
pca_df["outlier"] = df["outlier_prediction"].map({0:"Inlier", 1:"Outlier"})

print("Variance explained by PC1 + PC2:",
      f"{pca2.explained_variance_ratio_.sum()*100:.2f}%")
fig = px.scatter(pca_df, x="PC1", y="PC2", color="segment",
                 symbol="outlier", title="PCA projection: 12 features to 2 components",
                 opacity=0.75)
fig.show()

## 6. Feature contribution to principal components
Loadings help explain which original variables contribute most to each component.

In [ ]:
loadings = pd.DataFrame(
    pca2.components_.T,
    index=feature_cols,
    columns=["PC1_loading", "PC2_loading"]
)
display(loadings.sort_values("PC1_loading", key=abs, ascending=False))
plt.figure(figsize=(8, 6))
sns.heatmap(loadings, annot=True, cmap="vlag", center=0)
plt.title("PCA loadings")
plt.tight_layout()
plt.show()

## 7. Key findings (computed from this dataset)
Run this cell after the preceding analysis to print the headline metrics.

In [ ]:
max_abs_corr = corr.where(~np.eye(len(corr), dtype=bool)).abs().max().max()
print(f"Dataset records: {len(df):,}")
print(f"Number of numeric features: {len(feature_cols)}")
print(f"Isolation Forest outlier rate: {outlier_rate:.1f}%")
print(f"Maximum absolute pairwise correlation: {max_abs_corr:.3f}")
print(f"Variance retained by 2 PCA components: {pca2.explained_variance_ratio_.sum()*100:.1f}%")
print("PCA component variance:", np.round(pca2.explained_variance_ratio_*100, 2))

## Conclusion
This workflow demonstrates an end-to-end exploratory data analysis process: data quality checks, univariate distribution analysis, multivariate relationship analysis, unsupervised anomaly detection, and PCA-based dimensionality reduction.

**Important:** Report the exact values printed by the notebook. Because the dataset is synthetic, describe the project as a simulated analytical case study rather than claiming the findings came from a real-world business dataset.